# Wave point sensitivity check

`geo.nearest_era5arco()` returns the offshore ERA5 point ranked `idx` closest to a transect (0 = closest, 1 = 2nd closest, ...). This notebook checks how much that choice matters, by comparing `idx = 0, 1, 2` on:

1. the wave climate pulled from each point (Hs distribution, storm count), and
2. the full PCR model output (recession exceedance curves) once each point is calibrated and simulated.

The three points are usually a few km apart along the same offshore transect line, so this is mainly a check on how sensitive the pipeline is to the ERA5 grid resolution / point-selection tie-breaking, not a comparison of genuinely different locations.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import shapely
import hvplot.pandas  # noqa

from dotenv import load_dotenv

from pcr import io, geo, storm, builder
from pcr import visualisation as viz

load_dotenv()
cds_api_key = os.getenv('CDS-API-KEY')

IDX_VALUES = [0, 1, 2, 3]
COLORS = ['#2a78d6', '#e34948', "#1baf7a", "#eee01b"]  # one per idx, used consistently below

In [ ]:
import matplotlib.colors as mcolors 

# get color map 
cmap = plt.get_cmap('viridis_r')

# normalisation 
norm = mcolors.Normalize(vmin=min(IDX_VALUES), vmax=max(IDX_VALUES)+1)

# hex color 
idx_to_color = [mcolors.to_hex(cmap(norm(idx))) for idx in IDX_VALUES]

print(idx_to_color)

## Area of interest
Same B-III bbox used in `17_input_geo.ipynb`; swap for whichever AOI you want to sensitivity-check.

In [ ]:
BBOX = [81.809224, 7.412626, 81.850069, 7.460627]  # B-III

transect = builder.build_transect(BBOX)
transect

## The 3 nearest ERA5 points
Fetch the ERA5 buffer area once and reuse it for all three `idx` values, rather than re-hitting ARCO per point.

In [ ]:
ds_area = io.era5arco_area([transect.lon, transect.lat], cds_api_key, buffer=1)

points = {idx: geo.nearest_era5arco(transect, ds_area, idx=idx) for idx in IDX_VALUES}
points

In [ ]:
# map of transect vs. the 3 candidate points
gdf_points = gpd.GeoDataFrame(
    {'idx': IDX_VALUES},
    geometry=[shapely.Point(points[idx]) for idx in IDX_VALUES],
    crs='EPSG:4326',
)

# cast to string so hvplot treats it as categorical, not continuous
gdf_points['idx'] = gdf_points['idx'].astype(str)

# build a dict mapping each category -> color instead of a list
idx_to_color = {str(idx): mcolors.to_hex(cmap(norm(idx))) for idx in IDX_VALUES}

gdf_transect = gpd.GeoDataFrame([transect], crs='EPSG:4326')

(
    gdf_transect.hvplot(geo=True, tiles='EsriImagery', width=600, height=450, color='yellow', line_width=3, label='transect')
    * gdf_points.hvplot.points(
        geo=True,
        hover_cols=['idx'],
        color='idx',
        cmap=idx_to_color,
        size=120,
        label='ERA5 points',
        colorbar=False,   # suppress the colorbar
        legend=True,      # show discrete legend instead
      )
)

## Wave time series per point

In [ ]:
wave_by_idx = {}
for idx in IDX_VALUES:
    hs, dir_, tp, time, record_years, lon, lat = io.import_era5arco(transect, cds_api_key, idx=idx, ds=ds_area)
    wave_by_idx[idx] = dict(hs=hs, dir=dir_, tp=tp, time=time, record_years=record_years, lon=lon, lat=lat)
    print(f'idx={idx}: lon={lon:.4f}, lat={lat:.4f}, n_records={len(hs)}, record_years={record_years}')

## Wave climate comparison
Summary stats and detected storms (same thresholds `PCRModel` uses by default: `ts_hs=95`, `ts_dur=12.0`, `ts_between=48.0`) at each point.

In [ ]:
from pcr import erosion

rows = []
storm_by_idx = {}

for idx in IDX_VALUES:
    w = wave_by_idx[idx]
    detected, ts_storm = storm.detect(w['hs'], w['dir'], w['tp'], w['time'], ts_hs=95, ts_dur=12.0, ts_between=48.0)
    # storm check
    ts_storm = ts_storm.assign(hs2=ts_storm['hs']**2)
    dt = (ts_storm.iloc[1]['time'] - ts_storm.iloc[0]['time']) * 24

    # erosion check 
    v, _ = erosion.vector_mendoza(
        hss=detected['hs_max'], 
        tps=detected['tp_mean'], 
        durs=detected['duration']
    )

    storm_by_idx[idx] = dict(hs_max=detected['hs_max'].values, duration=detected['duration'].values, gap=detected['gap'].values, tp=detected['tp_mean'].values)
    rows.append({
        'idx': idx,
        'lon': w['lon'], 'lat': w['lat'],
        'Hs_mean': np.mean(w['hs']), 'Hs95': np.percentile(w['hs'], 95), 'Hs_max': np.max(w['hs']),
        'n_storms': len(detected), 'storms_per_year': len(detected) / w['record_years'],
        'dir_mean': np.mean(w['dir']), 'tp_mean': np.mean(w['tp']),
        'E_tot': 1 / 16 * 1025 * 9.81 * dt * np.sum(ts_storm['hs2']) / 1e6, 
        'V_tot': np.sum(v)
    })

df_wave_compare = pd.DataFrame(rows).set_index('idx')
df_wave_compare

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for idx, color in zip(IDX_VALUES, idx_to_color):
    ax.hist(wave_by_idx[idx]['hs'], bins=60, histtype='step', density=True, color=color, linewidth=1.5, label=f'idx={idx}')
ax.set_xlabel('Hs (m)')
ax.set_ylabel('Density')
ax.set_title('Significant wave height distribution by ERA5 point rank')
ax.legend()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for idx, color in zip(IDX_VALUES, COLORS):
    ax.hist(storm_by_idx[idx]['hs_max'], bins=60, histtype='step', density=True, color=color, linewidth=1.5, label=f'idx={idx}')
ax.set_xlabel('Hs (m)')
ax.set_ylabel('Density')
ax.set_title('Peak storm wave height distribution by ERA5 point rank')
ax.legend()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for idx, color in zip(IDX_VALUES, idx_to_color):
    ax.hist(storm_by_idx[idx]['tp'], bins=60, histtype='step', density=True, color=color, linewidth=1.5, label=f'idx={idx}')
ax.set_xlabel('Mean peak period (s)')
ax.set_ylabel('Density')
ax.set_title('Mean peak period distribution by ERA5 point rank')
ax.legend()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for idx, color in zip(IDX_VALUES, idx_to_color):
    ax.scatter(storm_by_idx[idx]['tp'], storm_by_idx[idx]['hs_max'], color = color, label=f'idx={idx}')
ax.set_xlabel('Tp (s)')
ax.set_ylabel('Hs (m)')
ax.set_title('Peak storm wave height distribution by ERA5 point rank')
ax.legend()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for idx, color in zip(IDX_VALUES, idx_to_color):
    ax.hist(storm_by_idx[idx]['gap'], bins=60, histtype='step', density=True, color=color, linewidth=1.5, label=f'idx={idx}')
ax.set_xlabel('Gap (days)')
ax.set_ylabel('Density')
# ax.set_xlim([0,200])
ax.set_title('Peak storm wave height distribution by ERA5 point rank')
ax.legend()

## Full PCR model per point
Each point gets its own `rec_rate` calibration (the lookup table in `pcr.calibration` is keyed on `lon_wave`/`lat_wave`, so idx=0/1/2 won't share a cached value). The RNG seed is reset before each run so the three simulations draw the same synthetic storm sequence, isolating differences to the wave data itself rather than Monte Carlo noise.

`NR_SIMULATION` is kept modest here for a quick check; bump it up for a final comparison.

In [ ]:
YEAR_START = 2020
YEAR_END = 2119
NR_SIMULATION = 50_000

models = {}
for idx in IDX_VALUES:
    print(f'\n=== idx={idx} ===')
    np.random.seed(42)
    models[idx] = builder.build_model(
        bbox=BBOX,
        idx=idx,
        ds=ds_area,
        calibrate=True,
        scenario='ssp126',
        wl0=0.0,
        rec_rate=None,
        m=0.024,
        c1 = 1.339, 
        c2 = 1.849,
        ts_hs=95,
        ts_dur=12.0,
        ts_between=48.0,
        year_start=YEAR_START,
        year_end=YEAR_END,
        nr_simulation=NR_SIMULATION,
        cds_api_key=cds_api_key
    )
    models[idx].run_simulation()

## Recession exceedance curves

In [ ]:
# idx=0 (closest point) as the reference line, shaded by the min/max spread across idx=0-2, at two horizon years
YEARS_CHECK = [25, 50, 75, 100]  # years after year_start
exceedance = np.linspace(0, 100, NR_SIMULATION)
color_year = ['k', 'k', 'r', 'r']
dashes = ['-', '--', '-', '--']

fig, ax = plt.subplots(figsize=(7, 5))
for color, dash, year_check in zip(color_year, dashes, YEARS_CHECK):
    row = year_check - 1
    sorted_by_idx = {idx: np.sort(-models[idx].shoreline_stats[row])[::-1] for idx in IDX_VALUES}
    stacked = np.vstack(list(sorted_by_idx.values()))
    lo, hi = stacked.min(axis=0), stacked.max(axis=0)

    ax.fill_betweenx(exceedance, lo, hi, color=color, alpha=0.15, label=f'1% range: {(hi-lo)[np.searchsorted(exceedance, 1)]:.2f} m')
    ax.plot(sorted_by_idx[0], exceedance, color=color, linestyle=dash, linewidth=1.8, label=f'closest wave: {year_check+2020}')

ax.set_yscale('log')
ax.set_ylim([1, 100])
ax.set_xlim([-40, 120])
ax.set_xlabel('Recession (m)')
ax.set_title(f'sensitivity to wave point choice')
ax.grid(True, which='both', alpha=0.3)
ax.legend()

ax.set_ylabel('Exceedance probability (%)')
fig.tight_layout()

## Summary table
Calibrated `rec_rate` plus median/p5/p95 recession at a few horizon years, per point.

In [ ]:
def summarize(model, years=[25, 50, 75, 100]):
    out = {}
    for y in years:
        vals = model.shoreline_stats[y - 1]
        out[f'median_R{y}'] = np.median(vals)
        out[f'p5_R{y}'] = np.percentile(vals, 5)
        out[f'p95_R{y}'] = np.percentile(vals, 95)
    return out

rows = []
for idx in IDX_VALUES:
    m = models[idx]
    row = {'idx': idx, 'lon': m.lon_wave, 'lat': m.lat_wave, 'rec_rate': m.rec_rate, 'rec_rate_m_per_yr': m.rec_rate * 365}
    row.update(summarize(m))
    rows.append(row)

df_summary = pd.DataFrame(rows).set_index('idx')
df_summary

Read the summary as: if `median_R100` (and the p5/p95 band) barely move across idx=0/1/2 relative to their own width, point selection isn't a meaningful source of uncertainty for this AOI — the rest of the Monte Carlo spread dominates. A large, consistent shift (e.g. one idx sitting systematically more/less exposed) would mean `nearest_era5arco`'s tie-breaking is worth revisiting for this coastline.